# 12 — Gate 4: the ensemble runner (spec v0.12 §7, §8, §10; VERSION switch added v0.17)

**VERSION = "v3"** (cell 1) solves the curated-EFG-block manifest `spec/manifest_v3.csv` (12 design formulations) into `analyses/y2y/runs_v3/` — anchors, MGA members and LP twins; the k-best pools (the E5 by-product, discharged) are skipped. `VERSION = "v1"` reproduces the as-frozen 2026-08-30 run (`runs/`). The v1 record is never overwritten.

Solves every frozen formulation in `spec/manifest.csv`, serial, **fully resumable** (each artifact
skipped when its output exists). Per formulation, into `analyses/y2y/runs/<formulation_id>/`:

| artifact | what | config |
|---|---|---|
| `kbest/` | engine run, k-best pool | Gurobi binary, opt_gap 1e-4, portfolio 50 @ 5% |
| `twin/` | engine run, LP twin | Gurobi **proportion** (the v0.10 twin ruling) |
| `anchor.tif` + `mga_g05.tif` | certified anchor + 50 MGA members | `mga_core`, g=5%, k=50 |

ssp245 formulations solve on the 245 macrorefugia realization (layer path patched before ingest;
recorded in each `formulation_meta.json`). Reference formulation: anchor/MGA exist from Gate 2b; kbest/twin
are manifest pointers to the Gate-2 record. ~45 min/formulation ⇒ **~10 h for the 13 open formulations**;
**live internet throughout** (WLS). Kernel `R (y2y)`.

In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)

# ---- VERSION switch (study plan v0.17; supersede, never delete) --------------------------------------------
# "v1" = the as-frozen 2026-08-30 record (spec/manifest.csv, runs/); "v3" = the curated EFG block re-solve
# (spec/manifest_v3.csv, runs_v3/; 12 design formulations). config.EFG_SUBDIR must match (asserted after ingest).
VERSION <- "v3.1"      # v3.1 = the curated block with window-derived targets (study plan v0.17.3)
MANIFEST_REL <- if (VERSION == "v1") "analyses/y2y/spec/manifest.csv" else sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION)
FREEZE_REL   <- if (VERSION == "v1") "analyses/y2y/spec/manifest_freeze.sha256" else sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL     <- if (VERSION == "v1") "analyses/y2y/runs" else sprintf("analyses/y2y/runs_%s", VERSION)
EFG_SUBDIR_EXPECTED <- if (VERSION == "v1") "iucn_efg" else paste0("iucn_efg_", sub("\\..*$", "", VERSION))   # minor versions share the block
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
for (col in c("kbest_ref", "twin_ref")) {           # v3 has no pool/twin pointers: an all-empty column reads as NA
  if (!col %in% names(MAN)) MAN[[col]] <- ""
  MAN[[col]] <- ifelse(is.na(MAN[[col]]), "", as.character(MAN[[col]]))
}
stopifnot(nrow(MAN) %in% c(12, 14))
# verify the freeze hash before solving against it
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot("manifest does not match its freeze hash -- STOP" =
            identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig))
cat(sprintf("VERSION %s: %s verified against freeze hash %s... (%d formulations)\n", VERSION, basename(MANIFEST_REL), substr(dig, 1, 16), nrow(MAN)))
RUNS <- file.path(PROJ, RUNS_REL)
DO_KBEST <- VERSION == "v1"        # v3 re-solves anchors, MGA and twins only (study plan v0.17)
REAL245 <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"

manifest refreshed from config.py (analysis=y2y)
VERSION v3.1: manifest_v3.1.csv verified against freeze hash 259f35eddcc33476... (12 formulations)


In [2]:
# ---- two ingested base contexts, built ONCE (one per climate level) ------------------------
ctx585 <- pr_setup(mpath, PROJ)
ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))

ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
efg_used <- ctx585$layers$path[ctx585$layers$role == "feature_efg"]
stopifnot("manifest.json enumerates a different EFG block than VERSION expects -- check config.EFG_SUBDIR" =
            length(efg_used) > 0 && all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), efg_used)))
cat(sprintf("EFG block: %d features from %s/\n", length(efg_used), EFG_SUBDIR_EXPECTED))
cat("base contexts ready (585 canonical; 245 with the realization layer patched)\n")

base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585

prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 features (8 continuous + 20 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 28 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 28 feature

In [3]:
# ---- DRY PLAN (no solves): the per-formulation worklist -------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cd <- file.path(RUNS, row$formulation_id)
  kb <- if (!DO_KBEST) "skip" else if (nzchar(row$kbest_ref)) "ref" else if (file.exists(file.path(cd, "kbest/run_summary.json"))) "done" else "TODO"
  tw <- if (nzchar(row$twin_ref))  "ref" else if (file.exists(file.path(cd, "twin/run_summary.json")))  "done" else "TODO"
  mg <- if (file.exists(file.path(cd, "mga_g05.tif"))) "done" else "TODO"
  cat(sprintf("%-22s %-9s kbest:%-5s twin:%-5s mga:%-5s\n",
              row$formulation_id, sub("_2071_2100", "", row$climate_level), kb, tw, mg))
}

s0_ssp585_theta5       ssp585    kbest:skip  twin:TODO  mga:TODO 
s1_ssp585_theta5       ssp585    kbest:skip  twin:TODO  mga:TODO 
s2_ssp585_theta5       ssp585    kbest:skip  twin:TODO  mga:TODO 
s3_ssp585_theta5       ssp585    kbest:skip  twin:TODO  mga:TODO 
s4_ssp585_theta3       ssp585    kbest:skip  twin:TODO  mga:TODO 
s5_ssp585_theta5       ssp585    kbest:skip  twin:TODO  mga:TODO 
s0_ssp245_theta5       ssp245    kbest:skip  twin:TODO  mga:TODO 
s1_ssp245_theta5       ssp245    kbest:skip  twin:TODO  mga:TODO 
s2_ssp245_theta5       ssp245    kbest:skip  twin:TODO  mga:TODO 
s3_ssp245_theta5       ssp245    kbest:skip  twin:TODO  mga:TODO 
s4_ssp245_theta3       ssp245    kbest:skip  twin:TODO  mga:TODO 
s5_ssp245_theta5       ssp245    kbest:skip  twin:TODO  mga:TODO 


In [4]:
# ---- runner helpers ------------------------------------------------------------------------
form_wt <- function(row) list(w = jsonlite::fromJSON(row$weight_vector),
                              t = jsonlite::fromJSON(row$target_vector))

run_engine_artifact <- function(row, artifact, ov) {
  cd_rel <- file.path(RUNS_REL, row$formulation_id)
  done <- file.path(PROJ, cd_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s/%s exists -- skipped\n", row$formulation_id, artifact)); return(invisible(NULL)) }
  wt <- form_wt(row)
  actx <- do.call(pr_override, c(list(base_for(row),
      targets                    = wt$t,
      feature_weight_multipliers = wt$w,
      results_dir                = cd_rel,
      results_subdir             = artifact), ov))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  invisible(NULL)
}

run_mga_artifact <- function(row) {
  cd <- file.path(RUNS, row$formulation_id)
  if (file.exists(file.path(cd, "mga_g05.tif"))) {
    cat(sprintf("   %s/mga exists -- skipped\n", row$formulation_id)); return(invisible(NULL)) }
  wt <- form_wt(row)
  actx <- pr_override(base_for(row),
      targets = wt$t, feature_weight_multipliers = wt$w,
      results_dir = file.path(RUNS_REL, row$formulation_id),
      results_subdir = "mga_build",
      solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
  gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested)
  mga_write(gen, cm, actx$cost, cd, "g05")
  r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
  v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
  terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U",
                     NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  jsonlite::write_json(list(
    formulation_id = row$formulation_id, estimator = row$estimator, verdict_rule = row$verdict_rule,
    anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap,
    anchor_runtime_s = anchor$runtime,
    macrorefugia_path = if (grepl("^ssp245", row$climate_level)) REAL245
                        else "input_data/aligned_stack/climate_type_macrorefugia.tif",
    weight_vector = form_wt(row)$w, target_vector = form_wt(row)$t,
    k = row$k_requested, g = row$band_gap_g,
    created_utc = format(Sys.time(), tz = "UTC")),
    file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  invisible(NULL)
}

In [6]:
# ---- THE LOOP: serial over the frozen formulations (resumable anywhere) --------------------------
t_batch <- proc.time()[["elapsed"]]
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
  if (!DO_KBEST) {
    cat("   kbest -> skipped (VERSION v3: anchors, MGA, twins only)\n")
  } else if (!nzchar(row$kbest_ref)) {
    run_engine_artifact(row, "kbest", list(solver = "gurobi", decision_type = "binary",
                                           opt_gap = row$opt_gap, portfolio_n = row$k_requested,
                                           portfolio_gap = row$band_gap_g))
  } else cat(sprintf("   kbest -> %s (Gate-2 record)\n", row$kbest_ref))
  if (!nzchar(row$twin_ref)) {
    run_engine_artifact(row, "twin", list(solver = "gurobi", decision_type = "proportion",
                                          opt_gap = row$opt_gap, portfolio_n = 1))
  } else cat(sprintf("   twin  -> %s (Gate-2 record, HiGHS exact)\n", row$twin_ref))
  # reference formulation: Gate 2b wrote gate2b_meta.json before the naming settled --
  # derive the standard meta file once so 13 reads every formulation uniformly
  cd <- file.path(RUNS, row$formulation_id)
  g2b <- file.path(cd, "gate2b_meta.json")
  fmeta <- file.path(cd, "formulation_meta.json")
  if (!file.exists(fmeta) && file.exists(g2b)) {
    m <- jsonlite::read_json(g2b)
    m$formulation_id <- row$formulation_id
    jsonlite::write_json(m, fmeta, auto_unbox = TRUE, pretty = TRUE, digits = 10)
    cat("   formulation_meta.json derived from gate2b_meta.json\n")
  }
  run_mga_artifact(row)
  cat(sprintf("== %s done | batch elapsed %.1f h\n", row$formulation_id,
              (proc.time()[["elapsed"]] - t_batch) / 3600))
}
cat("\nENSEMBLE COMPLETE -- next: analyses/y2y/13_gate4_analysis.ipynb\n")


===================== s0_ssp585_theta5 (1/12) =====================
   kbest -> skipped (VERSION v3: anchors, MGA, twins only)
   s0_ssp585_theta5/twin exists -- skipped
   s0_ssp585_theta5/mga exists -- skipped
== s0_ssp585_theta5 done | batch elapsed 0.0 h

===================== s1_ssp585_theta5 (2/12) =====================
   kbest -> skipped (VERSION v3: anchors, MGA, twins only)
   s1_ssp585_theta5/twin exists -- skipped
   s1_ssp585_theta5/mga exists -- skipped
== s1_ssp585_theta5 done | batch elapsed 0.0 h

===================== s2_ssp585_theta5 (3/12) =====================
   kbest -> skipped (VERSION v3: anchors, MGA, twins only)
   s2_ssp585_theta5/twin exists -- skipped
   s2_ssp585_theta5/mga exists -- skipped
== s2_ssp585_theta5 done | batch elapsed 0.0 h

===================== s3_ssp585_theta5 (4/12) =====================
   kbest -> skipped (VERSION v3: anchors, MGA, twins only)
   s3_ssp585_theta5/twin exists -- skipped
   s3_ssp585_theta5/mga exists -- skipped
== s3_s

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 1.752866)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.865468 (bound 4.865438, gap 6.35e-06) | 381,874 selected | 207 s
band wall appended: obj0 . x <= 5.108742  (g = 0.05 on z* = 4.865468)
g=0.05 iter 01/50: band 5.108741 (+5.00% of z*) OK | ham(anchor) 374,594 | 42 s
g=0.05 iter 02/50: band 5.108739 (+5.00% of z*) OK | ham(anchor) 332,590 | 39 s
g=0.05 iter 03/50: band 5.108741 (+5.00% of z*) OK | ham(anchor) 285,332 | 45 s
g=0.05 iter 04/50: band 5.108733 (+5.00% of z*) OK | ham(anchor) 233,810 | 44 s
g=0.05 iter 05/50: band 5.108738 (+5.00% of z*) OK | ham(anchor) 186,736 | 44 s
g=0.05 iter 06/50: band 5.108736 (+5.00% of z*) OK | ham(anchor) 303,972 | 44 s
g=0.05 iter 07/50: band 5.108740 (+5.00% of z*) OK | ham(anchor) 287,262 | 43 s
g=0.05 iter 08/50: band 5.108742 (+5.00% of z*) OK | ham(anchor) 263,254 | 40 s
g=0.05 iter 09/50: band 5.108705 (+5.00% of z*) OK | ham(anchor) 237,208 | 44 s
g=0.05 iter 10/50: band 5.108741 (

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 2.864201)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 29 rows, 1272942 columns and 15032935 nonzeros (Min)
Model fingerprint: 0xf1dba48f
Model has 28 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [5e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+04, 4e+05]

Presolve removed 12 rows a

...

compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 4.486618 (bound 4.486526, gap 2.04e-05) | 381,874 selected | 51 s
band wall appended: obj0 . x <= 4.710949  (g = 0.05 on z* = 4.486618)
g=0.05 iter 01/50: band 4.710948 (+5.00% of z*) OK | ham(anchor) 344,822 | 34 s
g=0.05 iter 02/50: band 4.710946 (+5.00% of z*) OK | ham(anchor) 276,740 | 36 s
g=0.05 iter 03/50: band 4.710948 (+5.00% of z*) OK | ham(anchor) 255,756 | 24 s
g=0.05 iter 04/50: band 4.710946 (+5.00% of z*) OK | ham(anchor) 241,012 | 36 s
g=0.05 iter 05/50: band 4.710948 (+5.00% of z*) OK | ham(anchor) 224,332 | 34 s
g=0.05 iter 06/50: band 4.710935 (+5.00% of z*) OK | ham(anchor) 211,564 | 40 s
g=0.05 iter 07/50: band 4.710948 (+5.00% of z*) OK | ham(anchor) 226,322 | 41 s
g=0.05 iter 08/50: band 4.710947 (+5.00% of z*) OK | ham(anchor) 254,152 | 43 s
g=0.05 iter 09/50: band 4.710948 (+5.00% of z*) OK | ham(anchor) 247,938 | 43 s
g=0.05 iter 10/50: band 4.710943 (+

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 10)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 29 rows, 1272942 columns and 15032935 nonzeros (Min)
Model fingerprint: 0x78db0108
Model has 28 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [5e-02, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+04, 4e+05]

Presolve removed 12 rows a

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (28 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.1 and 1)
││└•weights:    continuous values (between 0.05 and 10)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272942 cols (1272914 pu + 28 aux) x 29 rows | 191029 locked pu | modelsense min
anchor: objective 11.105506 (bound 11.105432, gap 6.68e-06) | 381,874 selected | 223 s
band wall appended: obj0 . x <= 11.660781  (g = 0.05 on z* = 11.105506)
g=0.05 iter 01/50: band 11.544878 (+3.96% of z*) OK | ham(anchor) 381,690 | 58 s
g=0.05 iter 02/50: band 11.654223 (+4.94% of z*) OK | ham(anchor) 381,690 | 43 s
g=0.05 iter 03/50: band 11.653824 (+4.94% of z*) OK | ham(anchor) 381,690 | 43 s
g=0.05 iter 04/50: band 11.649170 (+4.90% of z*) OK | ham(anchor) 381,690 | 40 s
g=0.05 iter 05/50: band 11.583323 (+4.30% of z*) OK | ham(anchor) 358,928 | 13 s
g=0.05 iter 06/50: band 11.605506 (+4.50% of z*) OK | ham(anchor) 307,416 | 12 s
g=0.05 iter 07/50: band 11.609547 (+4.54% of z*) OK | ham(anchor) 306,290 | 12 s
g=0.05 iter 08/50: band 11.619648 (+4.63% of z*) OK | ham(anchor) 308,426 | 12 s
g=0.05 iter 09/50: band 11.621398 (+4.65% of z*) OK | ham(anchor) 306,968 | 12 s
g=0.05 iter 10/50: ba

In [7]:
# ---- timing + integrity summary ------------------------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(RUNS, row$formulation_id)
  meta_f <- file.path(cd, "formulation_meta.json")
  kb_f <- if (nzchar(row$kbest_ref)) file.path(PROJ, row$kbest_ref, "run_summary.json")
          else file.path(cd, "kbest/run_summary.json")
  tw_f <- if (nzchar(row$twin_ref)) file.path(PROJ, row$twin_ref, "run_summary.json")
          else file.path(cd, "twin/run_summary.json")
  if (!file.exists(meta_f) || (DO_KBEST && !file.exists(kb_f)) || !file.exists(tw_f)) {
    cat(sprintf("%-22s INCOMPLETE\n", row$formulation_id)); next }
  m <- jsonlite::read_json(meta_f); tw <- jsonlite::read_json(tw_f)
  kb <- if (DO_KBEST) jsonlite::read_json(kb_f) else list(n_alternatives = NA_integer_, solve_seconds = NA_real_)
  tw_obj <- tryCatch(as.numeric(unlist(tw$solver_provenance$objective))[1], error = function(e) NA)
  ok <- if (is.na(tw_obj)) "?" else if (tw_obj <= m$anchor_objective + 1e-6) "OK" else "VIOLATED"
  cat(sprintf("%-22s anchor %.6f (gap %.0e, %4.0fs) | twin %.6f [LP<=MILP %s] | kbest %d sol %5.0fs\n",
              row$formulation_id, m$anchor_objective, m$anchor_gap, m$anchor_runtime_s,
              ifelse(is.na(tw_obj), NaN, tw_obj), ok, kb$n_alternatives, kb$solve_seconds))
}

s0_ssp585_theta5       anchor 4.902484 (gap 8e-06,  234s) | twin 4.902400 [LP<=MILP OK] | kbest NA sol    NAs
s1_ssp585_theta5       anchor 4.703443 (gap 2e-05,  168s) | twin 4.703300 [LP<=MILP OK] | kbest NA sol    NAs
s2_ssp585_theta5       anchor 5.065653 (gap 2e-05,  661s) | twin 5.065600 [LP<=MILP OK] | kbest NA sol    NAs
s3_ssp585_theta5       anchor 5.093379 (gap 1e-05,  349s) | twin 5.093300 [LP<=MILP OK] | kbest NA sol    NAs
s4_ssp585_theta3       anchor 4.524052 (gap 1e-05,   62s) | twin 4.524000 [LP<=MILP OK] | kbest NA sol    NAs
s5_ssp585_theta5       anchor 11.131507 (gap 7e-06,  257s) | twin 11.131400 [LP<=MILP OK] | kbest NA sol    NAs
s0_ssp245_theta5       anchor 4.865468 (gap 6e-06,  205s) | twin 4.865400 [LP<=MILP OK] | kbest NA sol    NAs
s1_ssp245_theta5       anchor 4.619475 (gap 8e-06,  140s) | twin 4.619400 [LP<=MILP OK] | kbest NA sol    NAs
s2_ssp245_theta5       anchor 5.056016 (gap 1e-05,  693s) | twin 5.055900 [LP<=MILP OK] | kbest NA sol    NAs
s3_ssp24

In [8]:
# ---- OPTIONAL: HiGHS spot-check twin (2nd cell; reference already has one) -----------------
# Flip to TRUE and run overnight if desired (worst observed HiGHS case: 109 min).
RUN_HIGHS_SPOTCHECK <- FALSE
if (RUN_HIGHS_SPOTCHECK) {
  row <- MAN[MAN$formulation_id == "s4_ssp585_theta3", ]
  run_engine_artifact(row, "twin_highs", list(solver = "highs", decision_type = "proportion",
                                              portfolio_n = 1))
}